# Módulo 02 · Aula 2: Busca híbrida e reordenação

**Duração:** 1h15

## Objetivos de aprendizagem

Ao final desta aula você será capaz de:

- reconhecer onde a busca vetorial falha: **códigos e nomes exatos** (SKUs como "NB-002", certificações como "IPX7",
  cupons como "BEMVINDO10");
- explicar e usar a busca lexical **BM25** (`rank_bm25`), incluindo a tokenização sem acentos;
- implementar a fusão **RRF** (Reciprocal Rank Fusion) e calcular um exemplo à mão;
- reordenar candidatos com o LLM (`rerank_with_llm`) sabendo o custo da chamada extra;
- usar o `HybridRetriever` do produto, um `BaseRetriever` do LangChain que encaixa em LCEL;
- **medir** vetorial, BM25 e híbrida no conjunto de avaliação antes de decidir qual usar.

## Pré-requisitos

- Aula 2.1 (`02_01_retrievers_and_filters`): retrievers, `k`, filtros por metadados, `vector_search` e
  `CUSTOMERS_ONLY`.

## O que vamos construir

```mermaid
flowchart LR
    Q([pergunta]) --> V[vetorial<br/>FAISS]
    Q --> B[BM25<br/>palavras exatas]
    V --> R[fusão RRF]
    B --> R
    R -.opcional.-> RR[reordenação<br/>LLM]
    R --> K[k trechos]
    RR --> K
```

As duas buscas usam o **mesmo filtro** de metadados (`CUSTOMERS_ONLY`). No fim, tudo isso é o `HybridRetriever` de
`src/rag/retrieval.py`, e vamos medir se ele vale a pena.

> Custo desta aula: **uma** chamada ao modelo de chat (reordenação) e cerca de 70 embeddings de consultas curtas
> (quase todos nas avaliações da seção 6).

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

In [2]:
from src.rag.retrieval import CUSTOMERS_ONLY, vector_search
from src.rag.vectorstore import index_exists, load_index, stored_chunks

if not index_exists():
    from src.rag.ingestion import index_folder

    index_folder()

index = load_index()
all_chunks = list(stored_chunks(index).values())
print(f"{len(all_chunks)} trechos no índice")


def label(doc) -> str:
    """Fonte curta de um trecho: arquivo e, nos PDFs, a página."""
    name = doc.metadata["source"].split("/")[-1]
    return name + (f"#p{doc.metadata['page']}" if doc.metadata.get("page") else "")


def show(docs):
    for position, doc in enumerate(docs, start=1):
        preview = doc.page_content.split("\n", 1)[-1][:70].replace("\n", " ")
        print(f"{position}. {label(doc):<42} {preview}")

2026-09-27 22:33:06 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


321 trechos no índice


## 1. Onde a busca vetorial falha: códigos e nomes exatos

Clientes (e atendentes) digitam códigos: o SKU que está na nota fiscal, a certificação que viram na caixa, o cupom que
receberam por e-mail. Veja a busca vetorial com uma consulta curta com o SKU do Notebook Nova Pro 16:

In [3]:
show(vector_search(index, "NB-002 bateria", k=4, metadata_filter=CUSTOMERS_ONLY))

1. nb-002_nova_pro_16.pdf#p3                  sempre na tomada. A perda gradual de capacidade é natural e não é defe
2. cx-001_nova_boom.pdf#p3                    TechNova | Caixa de Som Nova Boom (CX-001) | Manual do usuário Ação Co
3. cx-001_nova_boom.pdf#p2                    TechNova | Caixa de Som Nova Boom (CX-001) | Manual do usuário 1. Apre
4. cx-001_nova_boom.pdf#p2                    com a tampa da porta USB-C fechada Bateria Até 20 h (volume a 50%) Rec


**Observe:** o manual certo aparece em 1º, mas os outros três lugares são da **caixa de som Nova Boom** (CX-001), que
também tem "bateria" e um código parecido. Para o modelo de embeddings, "NB-002", "NB-001" e "CX-001" são sequências
raras de caracteres, sem significado próprio. Vamos medir a similaridade de cosseno entre alguns textos (uma chamada de
embeddings para os cinco):

In [4]:
import numpy as np
import pandas as pd

from src.models import get_embeddings

texts = ["NB-001", "NB-002", "CX-001", "Notebook Nova Pro 16", "bateria do NB-002"]
vectors = np.array(get_embeddings().embed_documents(texts))
pd.DataFrame(vectors @ vectors.T, index=texts, columns=texts).round(2)

,NB-001,NB-002,CX-001,Notebook Nova Pro 16,bateria do NB-002
NB-001,1.00,0.92,0.61,0.32,0.54
NB-002,0.92,1.00,0.57,0.30,0.59
CX-001,0.61,0.57,1.00,0.22,0.29
Notebook Nova Pro 16,0.32,0.30,0.22,1.00,0.40
bateria do NB-002,0.54,0.59,0.29,0.40,1.00


**Observe:** "NB-001" e "NB-002" têm cosseno de cerca de 0,92: para o modelo, são **quase o mesmo texto**, embora sejam
produtos diferentes. E "NB-002" fica mais longe de "Notebook Nova Pro 16" (o próprio produto) do que de "CX-001".
O embedding captura **significado**; um código não tem significado, tem **identidade**.

## 2. BM25: busca por palavras exatas

O **BM25** é o algoritmo clássico de busca por palavras (é o padrão do Elasticsearch e do OpenSearch). A nota de um
documento para uma consulta soma, para cada palavra da consulta:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{média}}\right)}$$

Em palavras:

| Ingrediente | Efeito |
|---|---|
| `f(t, d)`: quantas vezes a palavra aparece no trecho | mais ocorrências, nota maior, mas com saturação (`k1`) |
| `IDF(t)`: quão **rara** é a palavra na base | "nb-002" (rara) vale muito; "de" ou "bateria" (comuns) valem pouco |
| tamanho do trecho (`b`) | trechos longos não ganham só por terem mais palavras |

Antes de contar palavras, é preciso **tokenizar**: transformar o texto em uma lista de termos comparáveis. O produto
usa `tokenize` de `src/rag/retrieval.py`: minúsculas, **sem acentos**, só letras e números, preservando códigos com
hífen.

In [5]:
from src.rag.retrieval import tokenize

for text in ["Garantia do NB-002: não é à prova d'água (IPX7)!", "nao e a prova dagua", "Cupom BEMVINDO10 não aplicado"]:
    print(f"{text!r:<52} -> {tokenize(text)}")

"Garantia do NB-002: não é à prova d'água (IPX7)!"   -> ['garantia', 'do', 'nb-002', 'nao', 'e', 'a', 'prova', 'd', 'agua', 'ipx7']
'nao e a prova dagua'                                -> ['nao', 'e', 'a', 'prova', 'dagua']
'Cupom BEMVINDO10 não aplicado'                      -> ['cupom', 'bemvindo10', 'nao', 'aplicado']


**Observe:** "não" e "nao" viram o mesmo termo, e "NB-002" continua inteiro (não vira "nb" e "002"). Sem remover
acentos, quem digita sem acento (muito comum no celular) não encontraria nada. Repare também no limite: "d'água" vira
"d" e "agua", e "dagua" é outro termo. Tokenização é sempre um compromisso.

Agora o BM25 em uma base de brinquedo com seis frases, usando a biblioteca `rank_bm25`:

In [6]:
from rank_bm25 import BM25Okapi

toy_corpus = [
    "Notebook Nova Pro 16 (NB-002): bateria de 80 Wh, até 12 horas de uso.",
    "Caixa de som Nova Boom (CX-001): bateria para até 20 horas, certificação IPX7.",
    "Fone Nova Buds (FN-001): bateria de até 8 horas com cancelamento de ruído.",
    "Notebook Nova Air 14 (NB-001): tela de 14 polegadas e 1,2 kg.",
    "Política de troca: até 30 dias corridos após a entrega.",
    "Política de garantia: 12 meses para notebooks.",
]
toy_bm25 = BM25Okapi([tokenize(text) for text in toy_corpus])

print("IDF:", {term: round(toy_bm25.idf.get(term, 0), 2) for term in ["nb-002", "notebook", "bateria", "ipx7"]})
for query in ["NB-002 bateria", "bateria do notebook", "IPX7"]:
    scores = toy_bm25.get_scores(tokenize(query))
    print(f"{query:<22} notas: {np.round(scores, 2)}")

IDF: {'nb-002': 1.3, 'notebook': 0.59, 'bateria': 0.0, 'ipx7': 1.3}
NB-002 bateria         notas: [1.19 0.   0.   0.   0.   0.  ]
bateria do notebook    notas: [0.54 0.   0.   0.56 0.   0.  ]
IPX7                   notas: [0.   1.24 0.   0.   0.   0.  ]


**Observe:**

- "bateria" aparece em **metade** das frases: o IDF dela é 0 e ela não ajuda a separar nada. Quem decide a consulta
  "NB-002 bateria" é o termo raro "nb-002".
- "IPX7" encontra a única frase que tem o termo, com nota alta; as outras ficam com 0.
- Em "bateria do notebook", as duas frases de notebook pontuam por causa de "notebook". O BM25 não sabe que
  "notebooks" (plural, frase 6) é a mesma palavra: não há sinônimos nem variações.

## 3. BM25 na base da TechNova

O produto embala isso na classe `BM25Search`: ela tokeniza todos os trechos do índice uma vez e aplica o **mesmo
filtro de metadados** da busca vetorial.

In [7]:
from src.rag.retrieval import BM25Search

bm25 = BM25Search(all_chunks)

for query in ["NB-002 bateria", "IPX7", "BEMVINDO10"]:
    print(f"> {query}")
    show(bm25.search(query, k=4, metadata_filter=CUSTOMERS_ONLY))
    print()

> NB-002 bateria
1. nb-002_nova_pro_16.pdf#p2                  TechNova | Notebook Nova Pro 16 (NB-002) | Manual do usuário 1. Aprese
2. nb-002_nova_pro_16.pdf#p4                  TechNova | Notebook Nova Pro 16 (NB-002) | Manual do usuário aparece n
3. nb-002_nova_pro_16.pdf#p1                  TechNova Notebook Nova Pro 16 SKU: NB-002 Manual do usuário Notebook d
4. nb-002_nova_pro_16.pdf#p5                  TechNova | Notebook Nova Pro 16 (NB-002) | Manual do usuário 8. Cuidad

> IPX7
1. cx-001_nova_boom.pdf#p2                    TechNova | Caixa de Som Nova Boom (CX-001) | Manual do usuário 1. Apre
2. faq.csv                                    Pergunta: O Smartphone Nova X é à prova d'água? Resposta: Não. O Smart
3. faq.csv                                    Pergunta: A garantia cobre danos por água na Caixa de Som Nova Boom? R
4. cx-001_nova_boom.pdf#p1                    TechNova Caixa de Som Nova Boom SKU: CX-001 Manual do usuário Caixa de

> BEMVINDO10
1. coupon_not_applied.htm

**Observe:** para "NB-002 bateria", os quatro trechos agora são do manual certo. "IPX7" traz o manual da Nova Boom (a caixa
de som com essa certificação) e perguntas frequentes que citam o termo. "BEMVINDO10" devolve **um**
trecho só:
`BM25Search` descarta notas zero, e só um trecho tem o termo. Isso é uma vantagem (nenhum ruído) e um risco (se o
cliente errar uma letra do cupom, nada volta).

E o outro lado: quando o cliente usa **outras palavras** que as do documento, o BM25 se perde.

In [8]:
paraphrase = "Vocês atendem com pessoa de verdade no sábado? Até que horas?"

print("BM25:")
show(bm25.search(paraphrase, k=4, metadata_filter=CUSTOMERS_ONLY))
print("\nVetorial:")
show(vector_search(index, paraphrase, k=4, metadata_filter=CUSTOMERS_ONLY))

BM25:
1. invoice.html                               Não é possível emitir a nota em nome de terceiros.  Se você está compr
2. faq.csv                                    Pergunta: Vocês têm entrega expressa? Resposta: Sim. A modalidade Expr
3. business_purchases.html                    Compras para empresas com CNPJ  Empresas podem comprar na TechNova pel
4. business_purchases.html                    Crie uma senha com no mínimo 8 caracteres, incluindo letra maiúscula, 

Vetorial:


1. support_channels_and_hours.html            Canais e horários de atendimento  A TechNova atende pelo chat do site 
2. support_channels_and_hours.html            O que o assistente virtual resolve  O assistente virtual funciona o te
3. change_delivery_address.html               Com a sua concordância, a alteração é registrada e você recebe um núme
4. refund.md                                  ## Aprovação por um atendente humano   **Todo reembolso passa pela apr


**Observe:** o artigo certo é `support_channels_and_hours.html`. Ele não diz "pessoa de verdade"; diz "atendente
humano" (e "sábados", no plural, que para o BM25 é outro termo). A busca vetorial entende a equivalência e acerta em 1º
lugar; o BM25 nem o coloca entre os quatro.

| | Vetorial | BM25 |
|---|---|---|
| Sinônimos e paráfrases | ✅ | ❌ |
| Códigos, SKUs, cupons, nomes próprios | ⚠️ confunde parecidos | ✅ |
| Erros de digitação | ✅ tolera | ❌ termo diferente |
| Custo | um embedding por consulta | só CPU, sem API |

Cada busca falha onde a outra acerta. A **busca híbrida** roda as duas e junta os resultados.

## 4. Fusão RRF (Reciprocal Rank Fusion)

Somar as notas não funciona: a vetorial devolve **distâncias** (menor é melhor) e o BM25 devolve **pontos** (maior é
melhor, sem teto). O RRF ignora as notas e usa só as **posições**. Cada trecho ganha, em cada lista em que aparece:

$$\text{RRF}(d) = \sum_{\text{listas}} \frac{1}{60 + \text{posição}(d)}$$

A constante 60 (do artigo original, Cormack e outros, 2009) achata a diferença entre as primeiras posições: 1º vale
1/61 ≈ 0,0164 e 2º vale 1/62 ≈ 0,0161. Por isso **aparecer nas duas listas pesa mais do que ser 1º em uma só**.

Exemplo à mão: a vetorial devolveu `[A, B, C, D]` e o BM25 devolveu `[C, A, E, B]`.

| Trecho | Vetorial | BM25 | Conta | RRF |
|---|---|---|---|---|
| A | 1º | 2º | 1/61 + 1/62 | 0,03253 |
| C | 3º | 1º | 1/63 + 1/61 | 0,03227 |
| B | 2º | 4º | 1/62 + 1/64 | 0,03175 |
| E | | 3º | 1/63 | 0,01587 |
| D | 4º | | 1/64 | 0,01562 |

D foi o 4º da vetorial, mas terminou atrás de E: nenhum dos dois teve "segundo voto", e E estava mais bem posicionado.
Implementamos em poucas linhas e conferimos com `rrf_fusion(*rankings, k=None)` do produto, que faz o mesmo com listas
de `Document` e identifica cada trecho pelo `id` gravado na ingestão (o mesmo trecho vindo das duas buscas conta uma
vez):

In [9]:
from langchain_core.documents import Document

from src.rag.retrieval import rrf_fusion


def rrf(*rankings: list[str], constant: int = 60) -> list[tuple[str, float]]:
    """Reciprocal Rank Fusion: soma 1 / (constant + posição) de cada item em cada lista."""
    scores: dict[str, float] = {}
    for ranking in rankings:
        for position, item in enumerate(ranking, start=1):
            scores[item] = scores.get(item, 0.0) + 1 / (constant + position)
    return sorted(scores.items(), key=lambda pair: pair[1], reverse=True)


for item, score in rrf(["A", "B", "C", "D"], ["C", "A", "E", "B"]):
    print(f"{item}: {score:.5f}")


def as_docs(names: list[str]) -> list[Document]:
    return [Document(page_content=name, metadata={"id": name}) for name in names]


fused = rrf_fusion(as_docs(["A", "B", "C", "D"]), as_docs(["C", "A", "E", "B"]))
print("rrf_fusion do produto:", [doc.page_content for doc in fused])

A: 0.03252
C: 0.03227
B: 0.03175
E: 0.01587
D: 0.01562
rrf_fusion do produto: ['A', 'C', 'B', 'E', 'D']


> 💡 **Dica:** o RRF aceita qualquer número de listas (por exemplo, a mesma pergunta reescrita de três jeitos) e dá
> para dar **pesos** diferentes a cada lista, multiplicando a fração. Ele premia **consenso**: se as duas buscas
> concordarem no trecho errado, a fusão também erra (vamos ver um caso na seção 6.4).

## 5. Reordenação (rerank) com o LLM

As buscas são rápidas porque comparam a pergunta com cada trecho **separadamente** (um vetor, uma lista de palavras).
Um **reranker** lê a pergunta e o trecho **juntos** e julga se um responde a outra: é bem mais preciso, e bem mais
caro. Por isso ele só reordena os poucos candidatos que as buscas já trouxeram.

`rerank_with_llm(query, chunks, k)` faz isso com o nosso próprio LLM, em **uma** chamada: numera os candidatos, pede uma
nota de 0 a 3 para cada um (saída estruturada `ChunkScores`, com o prompt `RERANK_PROMPT` de `src/prompts.py`),
descarta as notas 0 e devolve os `k` melhores.

Vamos testar com a pergunta do sábado, em que a híbrida (que ainda vamos montar) traz o artigo certo só em 4º lugar:

In [10]:
import time

from langchain_core.messages.utils import count_tokens_approximately

from src.rag.retrieval import rerank_with_llm

candidates = rrf_fusion(
    vector_search(index, paraphrase, k=12, metadata_filter=CUSTOMERS_ONLY),
    bm25.search(paraphrase, k=12, metadata_filter=CUSTOMERS_ONLY),
    k=12,
)
print("Híbrida (RRF), 4 primeiros de 12 candidatos:")
show(candidates[:4])

start = time.perf_counter()
reranked = rerank_with_llm(paraphrase, candidates, k=4)
elapsed = time.perf_counter() - start

print("\nDepois da reordenação com o LLM:")
show(reranked)

rerank_context = "\n\n".join(c.page_content[:700] for c in candidates)
print(f"\nCusto: 1 chamada extra, ~{count_tokens_approximately([rerank_context])} tokens de entrada, {elapsed:.1f} s")

Híbrida (RRF), 4 primeiros de 12 candidatos:
1. change_delivery_address.html               Com a sua concordância, a alteração é registrada e você recebe um núme
2. track_order.html                           Se passaram mais de 24 horas sem nenhuma atualização, ou se o pedido u
3. business_purchases.html                    Trocas, garantia e reembolso  Compras de empresas seguem as mesmas pol
4. support_channels_and_hours.html            Canais e horários de atendimento  A TechNova atende pelo chat do site 
2026-09-27 22:33:08 | INFO     | src.models | Inicializando LLM | provedor=azure | deployment=gpt-5.4-mini



Depois da reordenação com o LLM:
1. change_delivery_address.html               Com a sua concordância, a alteração é registrada e você recebe um núme
2. support_channels_and_hours.html            Canais e horários de atendimento  A TechNova atende pelo chat do site 
3. track_order.html                           Reembolso integral:  o valor é devolvido na forma de pagamento origina
4. support_channels_and_hours.html            O que o assistente virtual resolve  O assistente virtual funciona o te

Custo: 1 chamada extra, ~1824 tokens de entrada, 1.9 s


**Observe:** o artigo de horários sobe de 4º para 2º (e os dois trechos dele ficam entre os quatro). O 1º lugar
continua com o trecho de alteração de endereço, que também cita atendente humano: notas empatadas mantêm a ordem da
fusão. O custo é uma
chamada inteira ao modelo **antes** da chamada que gera a resposta, com todos os candidatos no prompt.

| Reordenar com o LLM | Não reordenar |
|---|---|
| ordem mais precisa, descarta trechos inúteis | ordem só das buscas |
| +1 chamada, ~2 mil tokens e ~1 a 3 s por pergunta | sem custo extra |
| vale para bases grandes e `k` pequeno | suficiente quando a busca já acerta |

> 💡 **Dica:** em produção existem **rerankers dedicados** (modelos "cross-encoder", como os da Cohere ou o
> `bge-reranker`), mais baratos e rápidos que um LLM de chat para essa tarefa. A ideia é a mesma; o produto usa o LLM
> que já temos para não adicionar outra dependência. No `HybridRetriever`, a reordenação é **opcional** (`rerank=False`
> por padrão).

## 6. Do notebook para o projeto: [src/rag/retrieval.py](../src/rag/retrieval.py)

Tudo o que vimos está em `src/rag/retrieval.py`:

| Peça | Papel |
|---|---|
| `tokenize(text)` | minúsculas, sem acentos, códigos com hífen inteiros |
| `BM25Search(chunks)` | BM25 sobre os mesmos trechos do FAISS, com filtro de metadados |
| `vector_search(index, query, k, metadata_filter)` | busca vetorial com `fetch_k` folgado (aula 2.1) |
| `rrf_fusion(*rankings, k=None)` | fusão RRF por `id` do trecho |
| `rerank_with_llm(query, chunks, k)` | reordenação com uma chamada ao LLM |
| `HybridRetriever` | junta tudo em um retriever do LangChain |

### 6.1 `HybridRetriever` é um `BaseRetriever`

Para criar um retriever próprio no LangChain, basta herdar de `BaseRetriever` e implementar
`_get_relevant_documents(query)`. Em troca, a classe ganha `invoke`, `batch`, callbacks, tracing e composição com `|`,
como o `VectorStoreRetriever` da aula 2.1. Os campos de configuração:

| Campo | Padrão | Significado |
|---|---|---|
| `index` | (obrigatório) | o índice FAISS |
| `k` | `RETRIEVAL_K` = 4 | trechos devolvidos |
| `fetch_k` | `FETCH_K` = 12 | candidatos de **cada** busca antes da fusão e da reordenação |
| `mode` | `"hybrid"` | `"vector"`, `"bm25"` ou `"hybrid"` |
| `rerank` | `False` | reordenar com o LLM |
| `metadata_filter` | `CUSTOMERS_ONLY` | filtro de atualidade e de acesso |

In [11]:
from langchain_core.retrievers import BaseRetriever

from src.rag.retrieval import HybridRetriever

retriever = HybridRetriever(index=index)
print("É um BaseRetriever?", isinstance(retriever, BaseRetriever))
print(f"k={retriever.k} | fetch_k={retriever.fetch_k} | mode={retriever.mode} | rerank={retriever.rerank}")

show(retriever.invoke("NB-002 bateria"))

É um BaseRetriever? True
k=4 | fetch_k=12 | mode=hybrid | rerank=False


1. nb-002_nova_pro_16.pdf#p3                  sempre na tomada. A perda gradual de capacidade é natural e não é defe
2. nb-002_nova_pro_16.pdf#p2                  TechNova | Notebook Nova Pro 16 (NB-002) | Manual do usuário 1. Aprese
3. cx-001_nova_boom.pdf#p3                    TechNova | Caixa de Som Nova Boom (CX-001) | Manual do usuário Ação Co
4. nb-002_nova_pro_16.pdf#p4                  TechNova | Notebook Nova Pro 16 (NB-002) | Manual do usuário aparece n


`invoke` usa a configuração do objeto. O método `search(query, k=..., mode=..., rerank=...)` **sobrescreve** a
configuração só naquela chamada, sem criar outro retriever (e sem recalcular o BM25 dos 321 trechos). É assim que
comparamos os modos lado a lado:

In [12]:
comparison_queries = ["NB-002 bateria", paraphrase, "BEMVINDO10"]

rows = {}
for query in comparison_queries:
    for mode in ("vector", "bm25", "hybrid"):
        labels = [label(d) for d in retriever.search(query, mode=mode)]
        rows[(query[:30], mode)] = labels + ["(nada)"] * (retriever.k - len(labels))
pd.DataFrame(rows).T.rename(columns=lambda i: f"{i + 1}º")

1º  \
NB-002 bateria                 vector        nb-002_nova_pro_16.pdf#p3   
                               bm25          nb-002_nova_pro_16.pdf#p2   
                               hybrid        nb-002_nova_pro_16.pdf#p3   
Vocês atendem com pessoa de ve vector  support_channels_and_hours.html   
                               bm25                       invoice.html   
                               hybrid     change_delivery_address.html   
BEMVINDO10                     vector          coupon_not_applied.html   
                               bm25            coupon_not_applied.html   
                               hybrid          coupon_not_applied.html   

                                                                    2º  \
NB-002 bateria                 vector          cx-001_nova_boom.pdf#p3   
                               bm25          nb-002_nova_pro_16.pdf#p4   
                               hybrid        nb-002_nova_pro_16.pdf#p2   
Vocês atendem com pessoa de ve vector  support_channels_and_hours.html   
                               bm25                            faq.csv   
                               hybrid                 track_order.html   
BEMVINDO10                     vector                       payment.md   
                               bm25                             (nada)   
                               hybrid                       payment.md   

                                                                 3º  \
NB-002 bateria                 vector       cx-001_nova_boom.pdf#p2   
                               bm25       nb-002_nova_pro_16.pdf#p1   
                               hybrid       cx-001_nova_boom.pdf#p3   
Vocês atendem com pessoa de ve vector  change_delivery_address.html   
                               bm25         business_purchases.html   
                               hybrid       business_purchases.html   
BEMVINDO10                     vector       coupon_not_applied.html   
                               bm25                          (nada)   
                               hybrid       coupon_not_applied.html   

                                                                    4º  
NB-002 bateria                 vector          cx-001_nova_boom.pdf#p2  
                               bm25          nb-002_nova_pro_16.pdf#p5  
                               hybrid        nb-002_nova_pro_16.pdf#p4  
Vocês atendem com pessoa de ve vector                        refund.md  
                               bm25            business_purchases.html  
                               hybrid  support_channels_and_hours.html  
BEMVINDO10                     vector                       payment.md  
                               bm25                             (nada)  
                               hybrid                       payment.md

**Observe:** em "NB-002 bateria", a híbrida limpa quase todo o ruído da vetorial (três trechos do manual certo); em
"pessoa de verdade no sábado", ela fica **entre** as duas, com o artigo certo em 4º. A fusão não escolhe a melhor busca
para cada pergunta: ela faz uma média de posições. (Como todo retriever, ele também encaixa em chains LCEL com `|`,
como na aula 2.1.)

### 6.2 Medindo: vetorial x BM25 x híbrida

"A híbrida junta o melhor dos dois mundos" é uma hipótese, não um fato. Medimos com `evaluate_retrieval` de
`src/evaluation.py` (assunto da aula 3.4; aqui só usamos) nas 26 perguntas de `data/eval/questions.json` que têm
fonte esperada, nos `k=4` primeiros trechos:

| Métrica | O que mede |
|---|---|
| **taxa de acerto** (`hit_rate`) | em quantas perguntas pelo menos uma fonte esperada apareceu |
| **MRR** | média de 1 / posição da primeira fonte esperada (1,0 = sempre em 1º) |
| **cobertura** | fração das fontes esperadas que apareceram (perguntas com dois documentos) |

Custo: um embedding por pergunta nos modos `vector` e `hybrid` (52 no total); o BM25 roda só na CPU.

In [13]:
from src.evaluation import evaluate_retrieval

tables, summaries = {}, {}
for mode in ("vector", "bm25", "hybrid"):
    tables[mode], summaries[mode] = evaluate_retrieval(lambda q, mode=mode: retriever.search(q, mode=mode))

pd.DataFrame(summaries).T

,questions,hit_rate,mrr,coverage
vector,26.0,0.962,0.756,0.904
bm25,26.0,0.885,0.603,0.827
hybrid,26.0,0.923,0.683,0.904


**Observe:** nesta base e neste conjunto de perguntas, a **vetorial ganha**: taxa de acerto de 0,962 e MRR de 0,756,
contra 0,885 e 0,603 do BM25 e 0,923 e 0,683 da híbrida. A híbrida ficou **entre** as duas, não acima delas.

Por quê? As perguntas do conjunto são escritas como clientes escrevem ("pessoa de verdade", "pra", "tampinha"), cheias
de paráfrases, que é onde a vetorial brilha. O BM25 entra na fusão com listas piores e puxa trechos errados para cima.

### 6.3 Quem ganha em cada pergunta

Comparando o `rr` (1 / posição da primeira fonte esperada) pergunta a pergunta, só nas perguntas em que os modos
discordam:

In [14]:
per_question = tables["vector"][["id", "type", "question"]].copy()
for mode in tables:
    per_question[mode] = tables[mode]["rr"].round(2)

disagree = per_question[per_question[["vector", "bm25", "hybrid"]].nunique(axis=1) > 1]
display(disagree.assign(question=disagree["question"].str[:55]))

# Três casos em detalhe: as fontes dos 4 trechos em cada modo.
for qid in ["q06", "q09", "q02"]:
    row = per_question.index[per_question["id"] == qid][0]
    print(f"{qid}: {per_question.loc[row, 'question']}")
    print(f"     esperado: {tables['vector'].loc[row, 'expected']}")
    for mode in tables:
        retrieved = [s.split("/")[-1] for s in tables[mode].loc[row, "retrieved"]]
        print(f"     {mode:<7} rr={tables[mode].loc[row, 'rr']:.2f} {retrieved}")

,id,type,question,vector,bm25,hybrid
0,q01,fact,Qual o valor mínimo da compra pra ter frete gr...,0.50,0.33,0.50
1,q02,fact,Quanto tempo de garantia tem um notebook compr...,0.33,0.33,0.00
5,q06,fact,Posso mergulhar a caixa de som Nova Boom? Até ...,0.50,1.00,1.00
8,q09,fact,Vocês atendem com pessoa de verdade no sábado?...,1.00,0.00,0.25
12,q13,procedure,Como faço pra rastrear meu pedido?,1.00,0.33,1.00
14,q15,procedure,Meu fone parou de funcionar. Como aciono a gar...,0.33,0.00,0.33
18,q19,procedure,Coloquei o endereço errado no pedido. Dá pra m...,1.00,0.25,0.50
19,q20,procedure,Esqueci a senha da minha conta. Como recupero?,0.50,0.33,0.50
21,q22,multi_document,Se eu comprar o Headset Nova Studio no boleto ...,1.00,0.50,1.00
22,q23,multi_document,Minha Nova Boom caiu na piscina com a tampinha...,0.50,0.33,0.33


q06: Posso mergulhar a caixa de som Nova Boom? Até que profundidade?
     esperado: ['manuals/cx-001_nova_boom.pdf']
     vector  rr=0.50 ['faq.csv', 'cx-001_nova_boom.pdf', 'cx-001_nova_boom.pdf', 'cx-001_nova_boom.pdf']
     bm25    rr=1.00 ['cx-001_nova_boom.pdf', 'cx-001_nova_boom.pdf', 'cx-001_nova_boom.pdf', 'faq.csv']
     hybrid  rr=1.00 ['cx-001_nova_boom.pdf', 'cx-001_nova_boom.pdf', 'faq.csv', 'cx-001_nova_boom.pdf']
q09: Vocês atendem com pessoa de verdade no sábado? Até que horas?
     esperado: ['help_center/support_channels_and_hours.html']
     vector  rr=1.00 ['support_channels_and_hours.html', 'support_channels_and_hours.html', 'change_delivery_address.html', 'refund.md']
     bm25    rr=0.00 ['invoice.html', 'faq.csv', 'business_purchases.html', 'business_purchases.html']
     hybrid  rr=0.25 ['change_delivery_address.html', 'track_order.html', 'business_purchases.html', 'support_channels_and_hours.html']
q02: Quanto tempo de garantia tem um notebook comprado na Tech

**Observe:**

- **q06** ("mergulhar a caixa de som **Nova Boom**"): BM25 e híbrida põem o manual em 1º. O nome próprio do produto é
  um termo raro e exato, e a vetorial põe uma pergunta frequente na frente.
- **q09** ("pessoa de verdade no sábado"): a vetorial acerta em 1º, o BM25 erra e a híbrida fica no meio (4º).
- **q02** ("garantia de um notebook"): a vetorial acha a política em 3º; a híbrida a **perde**. A política é 3ª nas
  duas buscas, mas em **trechos diferentes** (seções diferentes), então nenhum ganha dois votos; a fusão intercala os
  1º e 2º lugares de cada lista, e a política cai para 5º.

> ⚠️ Repare que `faq.csv` aparece muito nos resultados e **não** conta como fonte esperada: às vezes ele até responde a
> pergunta, e a métrica penaliza. Toda métrica depende do gabarito; na aula 3.4 vamos discutir isso.

### 6.4 E nas consultas com códigos?

O conjunto oficial quase não tem consultas com SKU. Montamos um mini conjunto de seis consultas curtas, no formato do
`questions.json`, passamos em `questions=` e medimos também a **precisão**: quantos dos 4 trechos vêm do manual certo.

In [15]:
code_questions = [
    {"id": "c1", "type": "code", "question": "NB-002 bateria", "expected_sources": ["manuals/nb-002_nova_pro_16.pdf"]},
    {"id": "c2", "type": "code", "question": "CX-001 profundidade máxima", "expected_sources": ["manuals/cx-001_nova_boom.pdf"]},
    {"id": "c3", "type": "code", "question": "SM-004 carregador", "expected_sources": ["manuals/sm-004_nova_mini.pdf"]},
    {"id": "c4", "type": "code", "question": "MN-001 taxa de atualização HDMI", "expected_sources": ["manuals/mn-001_nova_view_27.pdf"]},
    {"id": "c5", "type": "code", "question": "FN-002 microfone", "expected_sources": ["manuals/fn-002_nova_studio.pdf"]},
    {"id": "c6", "type": "code", "question": "SM-001 resistência à água", "expected_sources": ["manuals/sm-001_nova_x.pdf"]},
]

code_tables, code_summaries = {}, {}
for mode in ("vector", "bm25", "hybrid"):
    code_tables[mode], code_summaries[mode] = evaluate_retrieval(
        lambda q, mode=mode: retriever.search(q, mode=mode), questions=code_questions
    )
    # Precisão: fração dos 4 trechos que vêm do manual certo (quanto ruído vai junto).
    code_summaries[mode]["precision"] = round(
        float(np.mean([r.count(e[0]) / len(r) for r, e in zip(code_tables[mode]["retrieved"], code_tables[mode]["expected"])])), 3
    )

display(pd.DataFrame(code_summaries).T)

# Precisão pergunta a pergunta: trechos do manual certo entre os 4.
pd.DataFrame(
    {mode: [f"{r.count(e[0])}/{len(r)}" for r, e in zip(t["retrieved"], t["expected"])] for mode, t in code_tables.items()},
    index=[q["question"] for q in code_questions],
)

,questions,hit_rate,mrr,coverage,precision
vector,6.0,0.833,0.833,0.833,0.625
bm25,6.0,1.000,1.000,1.000,0.833
hybrid,6.0,0.833,0.833,0.833,0.625


,vector,bm25,hybrid
NB-002 bateria,1/4,4/4,3/4
CX-001 profundidade máxima,4/4,3/4,4/4
SM-004 carregador,2/4,3/4,1/4
MN-001 taxa de atualização HDMI,4/4,3/4,3/4
FN-002 microfone,4/4,4/4,4/4
SM-001 resistência à água,0/4,3/4,0/4


**Observe:** aqui o jogo vira. O **BM25 acerta todas**, com pouco ruído. A vetorial erra "SM-001 resistência à água"
(traz a política de garantia, que fala de contato com líquidos) e, quando acerta, às vezes mistura manuais de outros
produtos ("NB-002 bateria": 1 de 4). A híbrida **empata** com a vetorial: limpa o ruído em "NB-002 bateria" (3 de 4),
piora em outras consultas (o BM25 puxa trechos de outros manuais e de `faq.csv` para a fusão) e herda o erro em
"SM-001": as duas buscas trazem a política de garantia, e o consenso vence.

### A lição: meça antes de adotar

| Tipo de consulta | Melhor modo nesta base |
|---|---|
| perguntas em linguagem natural, com paráfrases | vetorial |
| códigos, SKUs, cupons, nomes exatos | BM25 |
| mistura desconhecida | híbrida: raramente a melhor, raramente a pior |

O produto usa `mode="hybrid"` por padrão porque, no atendimento real, clientes colam códigos de produto e de cupom e
a híbrida evita o pior caso das duas. Mas os números mostram que essa não é uma escolha "óbvia": ela depende do
**tráfego real**. Por isso o modo é um parâmetro, e a decisão deve ser revista com um conjunto de avaliação que
represente as perguntas de verdade (aula 3.4).

## Resumo

- A busca vetorial entende **significado** e falha em **identidade**: "NB-001" e "NB-002" têm cosseno ~0,92.
- O **BM25** pontua palavras exatas, valorizando as **raras** (IDF). A tokenização sem acentos (`tokenize`) é parte da
  qualidade da busca. Ele não entende sinônimos.
- A **fusão RRF** soma `1 / (60 + posição)` de cada lista: usa posições, não notas, e premia o **consenso**.
- A **reordenação com o LLM** (`rerank_with_llm`) melhora a ordem ao custo de uma chamada extra por pergunta.
- `HybridRetriever` é um `BaseRetriever`: `invoke`, `batch`, `|` em LCEL; `search(query, k=..., mode=..., rerank=...)`
  sobrescreve a configuração em uma chamada.
- **Medido nesta base** (k=4): vetorial 0,962 / 0,756 (taxa de acerto / MRR), BM25 0,885 / 0,603, híbrida
  0,923 / 0,683. Nas consultas com códigos, o BM25 venceu. Híbrida não é automaticamente melhor: meça.

## Próxima aula

Temos os trechos. Na aula 2.3 o modelo passa a **responder** com eles: um prompt de RAG que usa só o contexto, cita as
fontes com `[n]` e admite quando não sabe, uma resposta estruturada (`RAGAnswer`) e a função `generate_answer`, que
transforma os números citados em fontes para mostrar ao cliente.